# Bab 11 - Neural Networks that Understand Language: Word Embeddings

**Nama:** Nanda Pratama ITM  
**NIM:** 101032330212  
**Kelas:** TK 47 01


> Notebook ini merupakan reproduksi edukatif dan pendalaman teori berdasarkan *Grokking Deep Learning* (Andrew W. Trask, Manning, 2019). Kode inti diadaptasi dari buku dan repositori pendamping resmi; perubahan kompatibilitas dijelaskan dalam dokumentasi.


## Capaian pembelajaran

Setelah menyelesaikan notebook ini, pembaca diharapkan mampu menjelaskan konsep utama bab, menghubungkannya dengan operasi NumPy, menjalankan contoh secara berurutan, dan menyebutkan batas interpretasinya.


## Representasi bahasa

Model tidak memahami teks mentah; kata harus dipetakan ke indeks atau vektor. One-hot menjaga identitas tetapi berdimensi besar dan tidak menyatakan kemiripan.


## Bag of words

Menjumlah atau merata-ratakan vektor kata mengabaikan urutan tetapi sering menjadi baseline kuat. Representasi ini cocok untuk sentimen ketika kehadiran kata lebih penting daripada susunannya.


## Embedding

Embedding adalah tabel vektor padat yang dipelajari. Kata yang menerima tekanan gradien serupa cenderung memiliki representasi serupa. Kedekatan embedding mencerminkan tujuan dan data training, bukan definisi universal makna.


## Negative sampling dan analogi

Negative sampling mengganti softmax seluruh kosakata dengan beberapa contoh negatif agar training lebih murah. Operasi vektor dapat menampilkan analogi, tetapi hasil sensitif terhadap frekuensi, bias korpus, normalisasi, dan metrik kemiripan.


## Reproduksi kode dan eksperimen

Jalankan sel dari atas ke bawah. `MODE_CEPAT=True` menjaga waktu eksekusi singkat; ubah ke `False` untuk konfigurasi yang lebih panjang.


In [1]:
from pathlib import Path
import numpy as np

np.set_printoptions(precision=4, suppress=True)
np.random.seed(1)
DATA_DIR = (Path.cwd().parent / "data").resolve()
MODE_CEPAT = True  # ubah ke False untuk eksperimen lebih panjang
assert DATA_DIR.is_dir(), f"Folder data tidak ditemukan: {DATA_DIR}"
print("Data:", DATA_DIR)
print("Mode cepat:", MODE_CEPAT)

def sigmoid(x):
    x = np.clip(x, -60, 60)
    return 1 / (1 + np.exp(-x))

def softmax(x):
    x = np.asarray(x)
    shifted = x - np.max(x, axis=-1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=-1, keepdims=True)


Data: D:\Kuliah\DL\UPLOAD_GITHUB\Grokking-Deep-Learning\data
Mode cepat: True


In [2]:
from collections import Counter
import gzip

limit = 2500 if MODE_CEPAT else 10000
with gzip.open(DATA_DIR / "reviews.txt.gz", "rt", encoding="utf-8") as stream:
    reviews = stream.read().splitlines()[:limit]
labels_text = (DATA_DIR / "labels.txt").read_text(encoding="utf-8").splitlines()[:limit]
labels = np.array([1.0 if label == "positive" else 0.0 for label in labels_text])
counts = Counter(word for review in reviews for word in review.split())
vocab = [word for word, _ in counts.most_common(2500 if MODE_CEPAT else 10000)]
word2index = {word: i for i, word in enumerate(vocab)}
token_ids = [[word2index[w] for w in set(r.split()) if w in word2index] for r in reviews]
print(len(reviews), len(vocab), token_ids[0][:10])


2500 2500 [143, 3, 0, 681, 28, 1807, 303, 1965, 51, 244]


In [3]:
rng = np.random.default_rng(11)
dim, alpha, epochs = 24, 0.05, (2 if MODE_CEPAT else 5)
embedding = rng.normal(0, 0.05, (len(vocab), dim))
output = rng.normal(0, 0.05, dim)

for epoch in range(epochs):
    for ids, target in zip(token_ids, labels):
        if not ids: continue
        hidden = embedding[ids].mean(axis=0)
        pred = float(sigmoid(hidden @ output))
        delta = pred - target
        old_output = output.copy()
        output -= alpha * delta * hidden
        embedding[ids] -= alpha * delta * old_output / len(ids)

predictions = [float(sigmoid(embedding[ids].mean(axis=0) @ output)) if ids else 0.5 for ids in token_ids]
print("training accuracy:", np.mean((np.array(predictions) >= 0.5) == labels))


training accuracy: 0.6064


In [4]:
def nearest(word, k=6):
    vector = embedding[word2index[word]]
    normed = embedding / (np.linalg.norm(embedding, axis=1, keepdims=True) + 1e-12)
    score = normed @ (vector / (np.linalg.norm(vector) + 1e-12))
    return [vocab[i] for i in np.argsort(score)[-k:][::-1]]

for word in [w for w in ["beautiful", "terrible", "excellent"] if w in word2index]:
    print(word, "->", nearest(word))


beautiful -> ['beautiful', 'older', 'told', 'dangerous', 'lou', 'job']
terrible -> ['terrible', 'issues', 'barely', 'saying', 'venoms', 'china']
excellent -> ['excellent', 'giving', 'suggest', 'efforts', 'wait', 'life']


## Interpretasi dan kesimpulan

Hasil harus dibaca bersama asumsi, konfigurasi, seed, serta data yang digunakan. Keberhasilan menjalankan kode tidak membuktikan generalisasi, kausalitas, keamanan, atau kelayakan deployment. Bandingkan hasil dengan teori di atas dan jelaskan setiap perbedaan sebelum mengubah konfigurasi.
